# AGENTIC TRACK — verifier-guided best-of-n + self-repair on HumanEvalFix

**Runtime: A100 or L4, ~1-1.5h, checkpointed every 20 problems (resumable).**

**⚠️ PROTOCOL DECLARATION — this is a NEW, SEPARATE track.** The frozen
single-shot headline (SFT v2: 38.6% mean / s3407 34.51%) is untouchable and
this notebook never competes with it. Here the model plays as an *agent*:
it may sample many candidate fixes, **execute the task's own provided unit
tests** in our sandbox, submit a verified fix, and on failure read the error
and retry (≤2 repair rounds). Metric: **verified-resolve rate** — the % of
164 problems where the agent submits a fix that passes the tests. Uses only
information the humanevalfixtests task provides (buggy code + tests, no
docstrings), with the model's NATIVE training prompt (prompt freedom is part
of the agentic protocol; labeled as such).

**Registered predictions (S2.35, BEFORE running):**
1. Calibration gate: gold solutions ≥160/164 pass the runner; buggy
   solutions ≥140/164 fail — else the plumbing is broken and the run is VOID.
2. Stage A (best-of-10, temp 0.8): verified resolve **45–52%** (the temp-0.2
   exam measured pass@10 = 47.2; diversity at 0.8 may add a little).
3. After 2 repair rounds: **52–62%**.
4. First-sample rate (single shot, temp 0.8, native prompt): genuinely
   uncertain, register band 28–40% — native format helps, hot temp hurts.

Model: SFT v2 seed 3407 (tracer lineage; flip SEED to 1234 later for the
best-seed run — 1234 scored 42.65 on the frozen exam).

In [ ]:
# 1) Setup
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time
PHASE8 = '/content/drive/MyDrive/rl-post-training/phase8'
SEED = 3407
CKPT = f'{PHASE8}/agentic_s{SEED}.json'
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
from prompts import build_repair_prompt, extract_code
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
from datasets import load_dataset
probs = list(load_dataset('bigcode/humanevalpack', 'python', split='test'))
assert len(probs) == 164
state = json.load(open(CKPT)) if os.path.exists(CKPT) else {}
print(len(probs), 'problems |', len(state), 'already in checkpoint')

In [ ]:
%%capture
!pip install unsloth
!pip uninstall -y -q torchao torchaudio torchvision timm

In [ ]:
# 2) Verifier + CALIBRATION GATE (hard — run is void if this fails)
import subprocess, tempfile
from concurrent.futures import ThreadPoolExecutor
STD_IMPORTS = ('from typing import List, Tuple, Dict, Set, Optional, Any\n'
               'import math, re, collections, itertools, functools, string\n')

def run_candidate(code_str, rec, timeout=10):
    prog = (STD_IMPORTS + code_str + '\n' + rec['test']
            + f"\ncheck({rec['entry_point']})\n")
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(prog); path = f.name
    try:
        r = subprocess.run([sys.executable, path], capture_output=True,
                           timeout=timeout)
        err = r.stderr.decode(errors='ignore')[-400:]
        return (r.returncode == 0, err if r.returncode != 0 else '')
    except subprocess.TimeoutExpired:
        return (False, 'TIMEOUT')
    finally:
        os.unlink(path)

def verify_many(codes, rec):
    with ThreadPoolExecutor(max_workers=8) as pool:
        return list(pool.map(lambda c: run_candidate(c, rec), codes))

gold_ok = buggy_fail = 0
for rec in probs:
    ok, _ = run_candidate(rec['declaration'] + rec['canonical_solution'], rec)
    gold_ok += ok
    ok, _ = run_candidate(rec['declaration'] + rec['buggy_solution'], rec)
    buggy_fail += (not ok)
print(f'CALIBRATION: gold pass {gold_ok}/164 (need >=160) | buggy fail {buggy_fail}/164 (need >=140)')
assert gold_ok >= 160, 'Runner plumbing broken — run VOID, do not proceed.'
assert buggy_fail >= 140, 'Tests not detecting bugs — run VOID, do not proceed.'
print('calibration gate PASSED')

In [ ]:
# 3) Model + generation helpers
import torch
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained(
    f'{PHASE8}/sft_v2_s{SEED}_ep2', max_seq_length=2048,
    load_in_4bit=True, dtype=None)
FastLanguageModel.for_inference(model)

def gen(prompt_text, k, temp=0.8, max_new=768):
    text = tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True)
    enc = tok([text], return_tensors='pt', padding=True,
              padding_side='left').to('cuda')
    with torch.no_grad():
        out = model.generate(**enc, do_sample=True, temperature=temp,
                             top_p=0.95, num_return_sequences=k,
                             max_new_tokens=max_new,
                             pad_token_id=tok.eos_token_id)
    return [extract_code(t) for t in tok.batch_decode(
        out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]

def base_prompt(rec):
    buggy = rec['declaration'] + rec['buggy_solution']
    return build_repair_prompt(buggy, [rec['test'] + f"\ncheck({rec['entry_point']})"])

REPAIR_TMPL = """{base}

A previous attempt:
```python
{prev}
```
It failed with:
```
{err}
```
Return only the corrected complete function in a Python code block."""
print('model ready (seed', SEED, ')')

In [ ]:
# 4) STAGE A — best-of-10 with test-verified selection (checkpointed)
N_INIT = 10
t0 = time.time()
for i, rec in enumerate(probs):
    tid = rec['task_id']
    if tid in state:
        continue
    codes = gen(base_prompt(rec), N_INIT)
    results = verify_many(codes, rec)
    n_pass = sum(ok for ok, _ in results)
    entry = {'n_pass_initial': n_pass,
             'first_sample_pass': bool(results[0][0]),
             'solved_stage': 'A' if n_pass > 0 else None}
    if n_pass > 0:
        entry['final_code'] = codes[[ok for ok, _ in results].index(True)]
    else:
        entry['fail_code'] = codes[0]
        entry['fail_err'] = results[0][1]
    state[tid] = entry
    if (i + 1) % 20 == 0 or i == len(probs) - 1:
        json.dump(state, open(CKPT, 'w'))
        done = len(state)
        solved = sum(1 for v in state.values() if v['solved_stage'])
        print(f'{done}/164  solved so far {solved} ({solved/done*100:.1f}%)  '
              f'[{(time.time()-t0)/60:.0f} min]')
json.dump(state, open(CKPT, 'w'))
sA = sum(1 for v in state.values() if v['solved_stage'] == 'A')
print(f'\nSTAGE A: {sA}/164 = {sA/164*100:.1f}% verified resolve')

In [ ]:
# 5) STAGE B — self-repair with execution feedback (2 rounds, k=4 each)
K_REPAIR, ROUNDS = 4, 2
for rnd in range(1, ROUNDS + 1):
    tag = f'R{rnd}'
    todo = [r for r in probs if state[r['task_id']]['solved_stage'] is None
            and f'tried_{tag}' not in state[r['task_id']]]
    print(f'--- round {rnd}: {len(todo)} unsolved ---')
    for i, rec in enumerate(todo):
        tid = rec['task_id']
        e = state[tid]
        prompt = REPAIR_TMPL.format(base=base_prompt(rec),
                                    prev=e['fail_code'], err=e['fail_err'])
        codes = gen(prompt, K_REPAIR)
        results = verify_many(codes, rec)
        e[f'tried_{tag}'] = True
        if any(ok for ok, _ in results):
            e['solved_stage'] = tag
            e['final_code'] = codes[[ok for ok, _ in results].index(True)]
        else:
            e['fail_code'] = codes[0]
            e['fail_err'] = results[0][1]
        if (i + 1) % 20 == 0:
            json.dump(state, open(CKPT, 'w'))
            print(f'  {i+1}/{len(todo)}')
    json.dump(state, open(CKPT, 'w'))
    s = sum(1 for v in state.values() if v['solved_stage'])
    print(f'after round {rnd}: {s}/164 = {s/164*100:.1f}%')

In [ ]:
# 6) VERDICT — the agentic table (separate protocol, labeled)
tot = len(state)
sA = sum(1 for v in state.values() if v['solved_stage'] == 'A')
s1 = sum(1 for v in state.values() if v['solved_stage'] == 'R1')
s2 = sum(1 for v in state.values() if v['solved_stage'] == 'R2')
first = sum(1 for v in state.values() if v['first_sample_pass'])
print('=' * 62)
print(f'AGENTIC TRACK — SFT v2 seed {SEED}, {tot} problems')
print('=' * 62)
print(f"{'single sample (temp 0.8, native prompt)':<44} {first/tot*100:6.1f}%")
print(f"{'Stage A: best-of-10, test-verified':<44} {sA/tot*100:6.1f}%")
print(f"{'+ repair round 1 (cumulative)':<44} {(sA+s1)/tot*100:6.1f}%")
print(f"{'+ repair round 2 (cumulative = FINAL)':<44} {(sA+s1+s2)/tot*100:6.1f}%")
print('-' * 62)
print('Context (DIFFERENT protocol — frozen single-shot exam, temp 0.2):')
print(f"{'  SFT v2 s3407 pass@1 / pass@10':<44}  34.51% / 47.24%")
print(f"{'  OctoCoder-16B pass@1':<44}  30.40%")
print(f"{'  GPT-4 pass@1 (paper)':<44} ~47.00%")
print('-' * 62)
print('Registered (S2.35): Stage A 45-52; final 52-62; single-sample 28-40.')
print('GUARDRAIL: the agentic number is NEVER reported as pass@1 and never')
print('enters the frozen table — always "verified-resolve, agentic protocol".')
print('\nBring the whole printout back.')

## Bring back to the session
The **verdict table** and the calibration line. Then optional: flip
`SEED = 1234` (best frozen seed, 42.65) and rerun for the best-seed agentic
number — the checkpoint file is per-seed, nothing collides.